In [91]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import numpy as np

## 1. Building Model Blocks

### 1.1 TDGCN MODEL

In [92]:
class TDGC(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 r: int = 8):
        super(TDGC, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        if in_channels == 3 or in_channels == 9:
            self.r_channels = 8
        else:
            self.r_channels = in_channels // r

        self.cdl_conv1 = nn.Conv2d(self.in_channels, self.r_channels, kernel_size=1) # 1x1 conv
        self.cdl_conv2 = nn.Conv2d(self.r_channels, self.out_channels, kernel_size=1) # 1x1 conv
        self.cfl_conv = nn.Conv2d(self.in_channels, self.out_channels, kernel_size=1) # 1x1 conv

        self.tanh = nn.Tanh()

        self.apply(self._weight_init)
    
    def _weight_init(self, module):
        if isinstance(module, nn.Conv2d):
            self._conv_init(module)
        if isinstance(module, nn.BatchNorm2d):
            self._bn_init(module, 1)

    def _conv_init(self, module):
        nn.init.kaiming_normal_(module.weight, mode='fan_out')
        nn.init.constant_(module.bias, 0)

    def _bn_init(self, module, scale):
        nn.init.constant_(module.weight, scale)
        nn.init.constant_(module.bias, 0)
    
    def forward(self, x, A=None, alpha=1, beta=1, gamma=0.1):
        # X:[C x T x N]; Batch size x in_channels x Time steps x Nodes
        # A:[N x N]; Nodes x Nodes

        '''CDL: Channel Decoupling Learning'''
        # 1 x 1 conv => [C/r x T x N]
        x1 = self.cdl_conv1(x)
        # Temporal Global Pooling => [C/r x N]
        x1 = x1.mean(-2)
        # Self-pairwise Subtraction => [C/r x N x N]
        x1 = x1.unsqueeze(-1) - x1.unsqueeze(-2)
        # Tanh activation
        x1 = self.tanh(x1)
        # 1 x 1 conv => [C' x N x N]
        x1 = self.cdl_conv2(x1)

        x1 = x1 * alpha + (A.unsqueeze(0).unsqueeze(0) if A is not None else 0)

        '''Coupling Feature Learning'''
        # 1 x 1 conv => [C' x T x N]
        x3 = self.cfl_conv(x)

        '''Temporal Decoupling Learning'''
        # Spatial Global Pooling => [T x N]
        x2 = x3.mean(-3)
        # Self-pairwise Subtraction => [T x N x N]
        x2 = x2.unsqueeze(-1) - x2.unsqueeze(-2)
        # Tanh activation
        x2 = self.tanh(x2)

        x1 = torch.einsum('ncuv,nctv->nctu', x1, x3)
        x3 = x3.permute(0, 2, 1, 3)
        x2 = torch.einsum('btmn,btcn->bctm', x2, x3)
        x1 = x1 * beta  + x2 * gamma 

        return x1 # X':[C' x T x N]; Batch size x out_channels x Time steps x Nodes

In [93]:
class TemporalConv(nn.Module):
    def __init__(self, 
                 in_channels: int,
                 out_channels: int,
                 kernel_size: int = 9,
                 stride: int = 1,
                 dilation: int = 1):
        
        super(TemporalConv, self).__init__()
        pad = int((kernel_size - 1) / 2) * dilation
        self.conv = nn.Conv2d(in_channels=in_channels,
                              out_channels=out_channels,
                              kernel_size=(kernel_size, 1),
                              padding=(pad, 0),
                              stride=(stride, 1),
                              dilation=(dilation, 1))
        
        self.bn = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        x = self.bn(self.conv(x))
        return x

In [94]:
class MultiScaleTemporalConv(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 kernel_size = 3,
                 stride: int = 1,
                 dilations: list = [1, 2, 3, 4], # this is the best combination selected after ablation study
                 residual: bool = True,
                 residual_kernel_size: int = 1):
        
        super(MultiScaleTemporalConv, self).__init__()

        assert out_channels % (len(dilations) + 2) == 0, "#out_channels must be divisible by #branches"

        self.num_branches = len(dilations) + 2
        branch_channels = out_channels // self.num_branches

        if type(kernel_size) == list:
            assert len(kernel_size) == len(dilations), "kernel_size and dilations must have the same length"
        else:
            kernel_size = [kernel_size] * len(dilations)

        '''Branch A'''
        self.branches = nn.ModuleList([
                            nn.Sequential(
                                nn.Conv2d(in_channels=in_channels,
                                            out_channels=branch_channels,
                                            kernel_size = 1,
                                            padding = 0), # 1x1 conv
                                nn.BatchNorm2d(branch_channels),
                                nn.ReLU(inplace=True),
                                TemporalConv(in_channels=branch_channels,
                                                out_channels=branch_channels,
                                                kernel_size=ks,
                                                stride=stride,
                                                dilation=dilation), # (k x 1 Conv, dilation = d) + BN
                            ) 
                            for ks, dilation in zip(kernel_size, dilations)
                        ])
        
        '''Branch C'''
        self.branches.append(nn.Sequential(
                                nn.Conv2d(in_channels=in_channels,
                                            out_channels=branch_channels,
                                            kernel_size=1,
                                            padding=0), # 1x1 conv
                                nn.BatchNorm2d(branch_channels), # BN
                                nn.ReLU(inplace=True), # ReLU
                                nn.MaxPool2d(kernel_size=(3, 1),
                                                stride=(stride, 1),
                                                padding=(1, 0)), # 3x1 MaxPool
                                nn.BatchNorm2d(branch_channels) # BN
                            ))
        
        '''Branch B'''
        self.branches.append(nn.Sequential(
                                nn.Conv2d(in_channels=in_channels,
                                            out_channels=branch_channels,
                                            kernel_size=1,
                                            padding=0,
                                            stride=(stride, 1)), # 1x1 conv
                                nn.BatchNorm2d(branch_channels) # BN
                            ))
        
        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = TemporalConv(in_channels=in_channels,
                                            out_channels=out_channels,
                                            kernel_size=residual_kernel_size,
                                            stride=stride)
            
        self.apply(self._weight_init)
    
    def _weight_init(self, module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode='fan_out')
            nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.normal_(module.weight, mean=1.0, std=0.02)
            nn.init.constant_(module.bias, 0)

    def forward(self, x):
        # X: (N, C, T, V); N: batch_size, C: in_channels, T: seq_len, V: num_joints
        res = self.residual(x)

        branches_out = []
        for branch in self.branches:
            branches_out.append(branch(x))

        out = torch.cat(branches_out, dim=1)
        out += res

        return out

In [95]:
class UnitTCN(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 kernel_size: int = 9,
                 stride: int = 1):
        
        super(UnitTCN, self).__init__()
        pad = int((kernel_size - 1) / 2)
        self.conv = nn.Conv2d(in_channels=in_channels,
                              out_channels=out_channels,
                              kernel_size=(kernel_size, 1),
                              padding=(pad, 0),
                              stride=(stride, 1))
        
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.apply(self._weight_init)

    def _weight_init(self, module):
        if isinstance(module, nn.Conv2d):
            self._conv_init(module)
        if isinstance(module, nn.BatchNorm2d):
            self._bn_init(module, 1)

    def _conv_init(self, conv):
        nn.init.kaiming_normal_(conv.weight, mode='fan_out')
        nn.init.constant_(conv.bias, 0)

    def _bn_init(self, bn, scale):
        nn.init.constant_(bn.weight, scale)
        nn.init.constant_(bn.bias, 0)

    def forward(self, x):
        x = self.bn(self.conv(x))
        return x

In [96]:
class UnitGCN(nn.Module):
    def __init__(self,
                 in_chanels: int,
                 out_channels: int,
                 A: torch.FloatTensor,
                 adaptive: bool = True,
                 residual: bool = True):
        
        super(UnitGCN, self).__init__()
        self.in_c = in_chanels
        self.out_c = out_channels
        self.adaptive = adaptive
        self.num_subset = A.shape[0]
        self.tdgc = nn.ModuleList()
        for i in range(self.num_subset):
            self.tdgc.append(TDGC(in_channels=in_chanels,
                                   out_channels=out_channels))
        
        if residual:
            if in_chanels != out_channels:
                self.residual = nn.Sequential(
                                nn.Conv2d(in_channels=in_chanels,
                                            out_channels=out_channels,
                                            kernel_size=1),
                                nn.BatchNorm2d(out_channels)
                            )
            else:
                self.residual = lambda x: x
        else:
            self.residual = lambda x: 0
        
        if self.adaptive:
            self.PA = nn.Parameter(torch.from_numpy(A.astype(np.float32)))
        else:
            self.A = Variable(torch.from_numpy(A.astype(np.float32)), requires_grad=False)

        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.alpha = nn.Parameter(torch.zeros(1))
        self.beta = nn.Parameter(torch.tensor(0.5))
        self.gamma = nn.Parameter(torch.tensor(0.1))

        self.apply(self._weight_init)
        
    def _weight_init(self, module):
        if isinstance(module, nn.Conv2d):
            self._conv_init(module)
        elif isinstance(module, nn.BatchNorm2d):
            self._bn_init(module, 1e-6)

    def _conv_init(self, conv):
        nn.init.kaiming_normal_(conv.weight, mode='fan_out')
        nn.init.constant_(conv.bias, 0)

    def _bn_init(self, bn, scale):
        nn.init.constant_(bn.weight, scale)
        nn.init.constant_(bn.bias, 0)

    def forward(self, x):
        y = None

        A = self.PA if self.adaptive else self.A.cuda(x.get_device())
        for i in range(self.num_subset):
            z = self.tdgc[i](x, A[i], self.alpha, self.beta, self.gamma)
            y = z + y if y is not None else z


        '''In paper, BN->ReLU->Residual'''
        y = self.bn(y)
        y += self.residual(x)
        y = self.relu(y)

        return y        

In [97]:
class unitTCNGCN(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 A: torch.FloatTensor,
                 stride: int = 1,
                 residual: bool = True,
                 adaptive: bool = True,
                 kernel_size: int = 5,
                 dilations: list = [1, 2]):
        
        super(unitTCNGCN, self).__init__()
        self.gcn1 = UnitGCN(in_chanels=in_channels,
                            out_channels=out_channels,
                            A=A,
                            adaptive=adaptive)
        self.tcn1 = MultiScaleTemporalConv(in_channels=out_channels,
                                            out_channels=out_channels,
                                            kernel_size=kernel_size,
                                            stride=stride,
                                            dilations=dilations,
                                            residual=False)
        self.relu = nn.ReLU(inplace=True)

        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = UnitTCN(in_channels=in_channels,
                                    out_channels=out_channels,
                                    kernel_size=1,
                                    stride=stride)
    
    def forward(self, x):
        y = self.relu(self.tcn1(self.gcn1(x)) + self.residual(x))
        return y

In [98]:
from utils.graph import Graph
import math

class TDGCN(nn.Module):
    def __init__(self,
                 num_class: int,
                 num_point: int,
                 num_person: int = 1,
                 graph: Graph = None,
                 in_channels: int = 3,
                 dropout: float = 0,
                 base_channels: int = 64,
                 final_embd_dim: int = 1024,
                 adaptive: bool = True):
        
        super(TDGCN, self).__init__()
        
        if graph is None:
            raise ValueError("Graph is required")
        
        self.graph = graph
        A = self.graph.A     
        
        self.num_class = num_class
        self.num_point = num_point

        self.data_bn = nn.BatchNorm1d(num_person * in_channels * num_point)

        base_channels = base_channels

        self.l1 = unitTCNGCN(in_channels=in_channels,
                                out_channels=base_channels,
                                A=A,
                                residual=False,
                                adaptive=adaptive)
        self.l2 = unitTCNGCN(in_channels=base_channels,
                                out_channels=base_channels,
                                A=A,
                                adaptive=adaptive)
        self.l3 = unitTCNGCN(in_channels=base_channels,
                                out_channels=base_channels,
                                A=A,
                                adaptive=adaptive)
        self.l4 = unitTCNGCN(in_channels=base_channels,
                                out_channels=base_channels,
                                A=A,
                                adaptive=adaptive)
        self.l5 = unitTCNGCN(in_channels=base_channels,
                                out_channels=base_channels * 2,
                                A=A,
                                stride=2,
                                adaptive=adaptive)
        self.l6 = unitTCNGCN(in_channels=base_channels * 2,
                                out_channels=base_channels * 2,
                                A=A,
                                adaptive=adaptive)
        self.l7 = unitTCNGCN(in_channels=base_channels * 2,
                                out_channels=base_channels * 2,
                                A=A,
                                adaptive=adaptive)
        self.l8 = unitTCNGCN(in_channels=base_channels * 2,
                                out_channels=base_channels * 4,
                                A=A,
                                stride=2,
                                adaptive=adaptive)
        self.l9 = unitTCNGCN(in_channels=base_channels * 4,
                                out_channels=base_channels * 4,
                                A=A,
                                adaptive=adaptive)
        self.l10 = unitTCNGCN(in_channels=base_channels * 4,
                                out_channels=base_channels * 4,
                                A=A,
                                adaptive=adaptive)
        
        self.fc = nn.Linear(base_channels * 4, final_embd_dim)
        self.fc1 = nn.Linear(final_embd_dim, num_class)

        if dropout > 0:
            self.drop = nn.Dropout(dropout)
        else:
            self.drop = lambda x: x

        self.apply(self._weight_init)

    def _weight_init(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, 0, math.sqrt(2./self.num_class))
            # nn.init.constant_(module.bias, 1)
        elif isinstance(module, nn.BatchNorm2d):
            self._bn_init(module, 1)

    def _conv_init(self, conv):
        nn.init.kaiming_normal_(conv.weight, mode='fan_out')
        nn.init.constant_(conv.bias, 0)

    def _bn_init(self, bn, scale):
        nn.init.constant_(bn.weight, scale)
        nn.init.constant_(bn.bias, 0)

    def forward(self, x):
        if len(x.shape) == 3:
            N, T, VC = x.shape # N: batch_size, T: seq_len, VC: num_joints * feature_dim
            x = x.view(N, T, self.num_point, -1).permute(0, 3, 1, 2).contiguous().unsqueeze(-1)

        N, C, T, V, M = x.shape # N: batch_size, C: in_channels, T: seq_len, V: num_joints, M: num_persons

        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * C * V, T) 
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)   

        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        x = self.l4(x)
        x = self.l5(x)
        x = self.l6(x)
        x = self.l7(x)
        x = self.l8(x)
        x = self.l9(x)
        x = self.l10(x)

        c_new = x.size(1)
        x = x.view(N, M, c_new, -1)
        x = x.mean(3).mean(1)

        x = self.drop(x)

        embed = self.fc(x)
        x = self.fc1(embed)

        return x, embed

### 1.3 SHASPEC MODEL

In [110]:
class SharedEncoderV1(nn.Module):
    def __init__(self,
                 d_model=512,
                 nhead=8,
                 num_encoder_layers=6):
        super(SharedEncoderV1, self).__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model,
                                                    nhead=nhead, 
                                                    batch_first=True, 
                                                    norm_first=True, 
                                                    activation='gelu')
        self.encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, 
                                                num_layers=num_encoder_layers)

        self.mixing_prob = 0.5

    def forward(self, x1, x2, mix=False):
        y1, y2 = None, None
        if x1 is None and x2 is None:
            Exception("Both inputs are None")
        elif x1 is None:
            y = self.encoder(x2.unsqueeze(1)).squeeze(1)
            y1, y2 = y, y
        elif x2 is None:
            y = self.encoder(x1.unsqueeze(1)).squeeze(1)
            y1, y2 = y, y
        else:
            '''Batch mixing'''
            if mix:
                B, _ = x1.shape
                idx = torch.from_numpy(np.array(np.where(np.random.binomial(1, self.mixing_prob, size=(B))))).to(x1.device)
                x1[idx], x2[idx] = x2[idx], x1[idx]

            x = torch.cat([x1.unsqueeze(1), x2.unsqueeze(1)], dim=1)
            y = self.encoder(x)
            y1, y2 = y[:, 0, :], y[:, 1, :]

        return y1, y2        

In [111]:
class ResidualFusion(nn.Module):
    def __init__(self, d_model=512):
        super(ResidualFusion, self).__init__()
        self.linear = nn.Linear(in_features=d_model*2, 
                                out_features=d_model)

    def forward(self, s, r):
        x = torch.cat((s, r), dim=-1)
        x = self.linear(x) + r
        
        return x

In [112]:
class ShaSpecV2(nn.Module):
    def __init__(self,
                 d_model=512,
                 nhead=8,
                 num_encoder_layers=6,
                 hidden_size=512,
                 dropout=0.1,
                 num_classes=2,
                 num_modalities=2):
        super(ShaSpecV2, self).__init__()
        self.shared_encoder = SharedEncoderV1(d_model=d_model,
                                            nhead=nhead,
                                            num_encoder_layers=num_encoder_layers)
        self.residual_fusion1 = ResidualFusion(d_model=d_model)
        self.residual_fusion2 = ResidualFusion(d_model=d_model)
        self.classifier = nn.Sequential(
            nn.Linear(in_features=d_model*num_modalities, 
                      out_features=hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_features=hidden_size, 
                      out_features=hidden_size//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_features=hidden_size//2, 
                      out_features=hidden_size//4),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size//4, 
                      out_features=num_classes)
        )

    def forward(self, x1, x2, mix=False):
        if x1 is None and x2 is None:
            Exception("Both inputs are None")

        if x1 is None:
            r1, r2 = self.shared_encoder(x1, x2, mix=mix)
            f1, f2 = r1, self.residual_fusion2(x2, r2)
            
        elif x2 is None:
            r1, r2 = self.shared_encoder(x1, x2, mix=mix)
            f1, f2 = self.residual_fusion1(x1, r1), r2

        else:
            r1, r2 = self.shared_encoder(x1, x2, mix=mix)
            f1, f2 = self.residual_fusion1(x1, r1), self.residual_fusion2(x2, r2)

        x = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)

        return x, r1, r2

### 1.4 PATCHTST MODEL

In [ ]:
import torch.nn as nn
from transformers import PatchTSTConfig, PatchTSTForClassification

class PatchTSTWrapper(nn.Module):
    def __init__(self,
                 input_channels=3,
                 num_targets=2,
                 context_length=1000,
                 patch_length=12,
                 stride=12,
                 use_cls_token=True):
        
        super(PatchTSTWrapper, self).__init__()
        self.config = PatchTSTConfig(
            num_input_channels=input_channels,
            num_targets=num_targets,
            context_length=context_length,
            patch_length=patch_length,
            stride=stride,
            use_cls_token=use_cls_token,
        )
        self.base = PatchTSTForClassification(config=self.config)

    def forward(self, x):
        """
        x: [bs x n_channels x d_model x num_patch]
        output: [bs x n_classes]
        """
        x = self.base.model(x)
        x = x.last_hidden_state[:, :, 0, :]

        x = self.base.head.flatten(x)
        embed = self.base.head.dropout(x)
        y = self.base.head.linear(embed)

        return y, embed
    

## 2. Models

In [113]:
# for graph type 1 -> hand in resting position
num_nodes = 21
inward = [(0, 1), (0, 5), (0, 9), (0, 13), (0, 17),
          (1, 2), (2, 3), (3, 4), (5, 6), (6, 7),
          (7, 8), (9, 10), (10, 11), (11, 12), (13, 14), (14, 15),
          (15, 16), (17, 18), (18, 19), (19, 20), (4, 8)]
outward = [(j, i) for i, j in inward]
self_link = [(i, i) for i in range(num_nodes)]

# for graph type 2 -> hand in motion
inward1 = [(4, 5), (8, 9), (12, 13), (16, 17), (13, 20)]
outward1 = [(j, i) for i, j in inward1]
inward2 = [(2, 4), (6, 8), (10, 12), (14, 16), (18, 20)]
outward2 = [(j, i) for i, j in inward2]

# for graph type 3 -> pose
num_nodes = 6
inward3 = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)]
outward3 = [(j, i) for i, j in inward3]
self_link3 = [(i, i) for i in range(num_nodes)]

In [114]:
from utils.graph import Graph

# graph type 1 -> hand in resting position
graph1 = Graph(num_node=21,
                self_link=self_link,
                inward=inward,
                outward=outward,
                neighbor=inward + outward)

# graph type 2 -> hand in motion
graph2 = Graph(num_node=21,
                self_link=self_link,
                inward=inward + inward1 + inward2,
                outward=outward + outward1 + outward2,
                neighbor=inward + outward + inward1 + inward2 + outward1 + outward2)

# graph type 3 -> pose
graph3 = Graph(num_node=6,
                self_link=self_link3,
                inward=inward3,
                outward=outward3,
                neighbor=inward3 + outward3)

In [ ]:
class GraphProcessor(nn.Module):
    def __init__(self,
                 num_classes=2,
                 dropout=0.1,
                 base_channels=64,
                 final_embd_dim=512):
        super(GraphProcessor, self).__init__()

        self.graph1 = TDGCN(num_class=num_classes,
                            num_point=21,
                            graph=graph1,
                            dropout=dropout,
                            base_channels=base_channels,
                            final_embd_dim=final_embd_dim)
        
        self.graph2 = TDGCN(num_class=num_classes,
                            num_point=21,
                            graph=graph2,
                            dropout=dropout,
                            base_channels=base_channels,
                            final_embd_dim=final_embd_dim)
        
        self.graph3 = TDGCN(num_class=num_classes,
                            num_point=6,
                            graph=graph3,
                            dropout=dropout,
                            base_channels=base_channels,
                            final_embd_dim=final_embd_dim)
        

    def forward(self, x1, graph_type):
        if graph_type[0] == 1:
            s1, embed = self.graph1(x1)
        elif graph_type[0] == 2:
            s1, embed = self.graph2(x1)
        elif graph_type[0] == 3:
            s1, embed = self.graph3(x1)
        else:
            Exception("Invalid graph type")
        
        return s1, embed
        

In [ ]:
class ModelV2(nn.Module):
    def __init__(self, config):
        super(ModelV2, self).__init__()
        self.config = config

        self.graph = GraphProcessor(num_classes=config.num_classes,
                                    dropout=config.dropout,
                                    final_embd_dim=config.final_embd_dim)
        
        self.sensor = PatchTSTWrapper(input_channels=3,
                                      num_targets=config.num_classes,
                                      context_length=1000,
                                      patch_length=12,
                                      stride=12,
                                      use_cls_token=True)
        
        if config.graph_pretrained is not None:
            self.graph.load_state_dict(torch.load(config.graph_pretrained, map_location=config.device))

        if config.sensor_pretrained is not None:
            self.sensor.load_state_dict(torch.load(config.sensor_pretrained, map_location=config.device))

        if config.graph_freeze:
            for param in self.graph.parameters():
                param.requires_grad = False
        
        if config.sensor_freeze:
            for param in self.sensor.parameters():
                param.requires_grad = False
        
        self.shaspec = ShaSpecV2(d_model=config.final_embd_dim,
                                 nhead=config.nhead,
                                 num_encoder_layers=config.num_encoder_layers,
                                 hidden_size=config.hidden_size,
                                 dropout=config.dropout,
                                 num_classes=config.num_classes,
                                 num_modalities=2)
        
    
    def forward(self, x1, x2, graph_type, mix=False):
        embed1, embed2 = None, None
        s1, s2 = None, None
        if x1 is not None:
            s1, embed1 = self.graph(x1, graph_type)
        if x2 is not None:
            s2, embed2 = self.sensor(x2)

        x, r1, r2 = self.shaspec(embed1, embed2, mix=mix)
        
        return x, s1, s2, r1, r2